# DEC5 000973: native tube-boundary evidence and support-guarded carving
One temporal frame, three held camera anchors. This is a failed repair control, not temporal validation. The input-quality audit separates zero/missing depth from surface evidence and verifies paired immutable inputs. No eval RGB enters prediction.
Native crops: `/mnt/data/lookcloser_dec5_5a3_surface_repair/supported_shell_control/review_{F,J,L}`.


In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
root = Path('/mnt/data/lookcloser_dec5_5a3_surface_repair')
findings = json.loads((root/'supported_shell_control/findings.json').read_text())
native = json.loads((root/'J_tube_boundary_trace/native_all62_v2.json').read_text())
assert findings['audit_status'] == 'input_and_comparison_integrity_pass_candidate_quality_fail'
assert findings['depth_maps_verified'] == native['train_camera_count'] == 62
assert len(native['points']) == len(native['missing_target_points']) == 3
assert not findings['temporal_promotion']


In [2]:
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1048576), b''):
            h.update(block)
    return h.hexdigest()
for path, expected in findings['retained_hashes'].items():
    assert digest(path) == expected, path
print('Verified retained inputs/outputs:', len(findings['retained_hashes']))


Verified retained inputs/outputs: 205


In [3]:
for point in native['points']:
    cameras = [s['physical_camera'] for s in point['sources']]
    assert len(cameras) == len(set(cameras)) == 62
    assert sum(point['counts'].values()) == 62
    strict = sum(s.get('near_within_0_001', 0) >= 20 for s in point['sources'])
    assert strict == point['strict_near_views']
    print(point['pixel'], 'strict near:', strict, 'free:', point['counts'].get('measured_free_space', 0))
assert [p['strict_near_views'] for p in native['points']] == [60, 61, 0]
for point in findings['point_receipts']:
    if point['index'] in (2, 3):
        assert not point['triangle_removed']
        assert point['before_filled_depth'] == point['after_filled_depth']
    else:
        assert point['triangle_removed']
        assert point['after_filled_depth'] > point['before_filled_depth'] + .04


[875, 310] strict near: 60 free: 0
[860, 310] strict near: 61 free: 0
[835, 335] strict near: 0 free: 19


In [4]:
evidence = np.load(root/'supported_shell_control/mesh/triangle_evidence.npz')
expected = (evidence['free_counts'] >= 3) & (evidence['near_counts'] <= 1)
assert np.array_equal(evidence['removed'], expected)
assert int(expected.sum()) == findings['triangles_carved'] == 4730
assert int(((evidence['free_counts'] >= 3) & (evidence['near_counts'] >= 2)).sum()) == 1188
for name in ('face_control', 'face_candidate'):
    assert set(findings[name]) == {'face_psnr', 'face_ssim', 'face_lpips'}
    assert np.isfinite(list(findings[name].values())).all()
    print(name, findings[name])
verdict = json.loads((root/'supported_shell_control/visual_review.json').read_text())
assert verdict['visual_pass'] == 0 and verdict['visual_fail'] == 3
assert len(verdict['inspected_crops']) == 7
assert all(row['status'] == 'fail' for row in verdict['anchors'])
print('Local shell correction is real; complete skin/fly-through quality still fails.')


face_control {'face_lpips': 0.04798922687768936, 'face_psnr': 28.724285125732422, 'face_ssim': 0.889098048210144}
face_candidate {'face_lpips': 0.04755343496799469, 'face_psnr': 28.753833770751953, 'face_ssim': 0.8893539905548096}
Local shell correction is real; complete skin/fly-through quality still fails.
